# Feature engineering — Credit Score Classification

Ce notebook transforme `data/train_cleaned.csv` (100 000 relevés mensuels, 8 mois
par client) en une table **une ligne par client** prête pour la modélisation :
`data/train_features.csv`.

**Plan :**
1. Création de nouvelles features (au niveau mensuel, dont la binarisation
   multi-label de `Type_of_Loan`)
2. Agrégation par `Customer_ID` (8 mois -> 1 ligne par client)
3. Encodage des variables catégorielles
4. Suppression des colonnes inutiles
5. Vérification finale et export


In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)


## 1. Chargement des données nettoyées


In [2]:
df = pd.read_csv('../data/train_cleaned.csv', low_memory=False)
print(f"Dimensions : {df.shape[0]} lignes x {df.shape[1]} colonnes")
print(f"Nombre de clients (Customer_ID) : {df['Customer_ID'].nunique()}")
print(f"Nombre de mois par client :")
print(df.groupby('Customer_ID').size().value_counts())


Dimensions : 100000 lignes x 29 colonnes
Nombre de clients (Customer_ID) : 12500
Nombre de mois par client :
8    12500
Name: count, dtype: int64


## 2. Création des nouvelles features

Ces ratios sont calculés **au niveau mensuel** (une valeur par ligne), avant
l'agrégation par client :

- `Ratio_Dette_Revenu` = `Outstanding_Debt` / `Annual_Income` — poids de la dette
  par rapport au revenu annuel.
- `Ratio_EMI_Salaire` = `Total_EMI_per_month` / `Monthly_Inhand_Salary` — part du
  salaire mensuel déjà engagée dans des mensualités de crédit.
- `Taux_Investissement` = `Amount_invested_monthly` / `Monthly_Inhand_Salary` —
  part du salaire mensuel investie.
- `Nb_produits` = `Num_Bank_Accounts` + `Num_Credit_Card` + `Num_of_Loan` —
  nombre total de produits bancaires détenus.

`Annual_Income` et `Monthly_Inhand_Salary` sont toujours strictement positifs
dans le dataset nettoyé (minimums de 7 006 et 304 respectivement), donc aucune
division par zéro n'est attendue ; on neutralise malgré tout par sécurité les
éventuelles valeurs infinies.


In [3]:
df['Ratio_Dette_Revenu'] = df['Outstanding_Debt'] / df['Annual_Income']
df['Ratio_EMI_Salaire'] = df['Total_EMI_per_month'] / df['Monthly_Inhand_Salary']
df['Taux_Investissement'] = df['Amount_invested_monthly'] / df['Monthly_Inhand_Salary']
df['Nb_produits'] = df['Num_Bank_Accounts'] + df['Num_Credit_Card'] + df['Num_of_Loan']

NOUVELLES_FEATURES = ['Ratio_Dette_Revenu', 'Ratio_EMI_Salaire', 'Taux_Investissement', 'Nb_produits']

# Neutralisation défensive des divisions par zéro / valeurs infinies éventuelles
n_infinies = 0
for col in NOUVELLES_FEATURES:
    inf_mask = np.isinf(df[col])
    n_infinies += int(inf_mask.sum())
    df.loc[inf_mask, col] = np.nan

print(f"Valeurs infinies neutralisées : {n_infinies}")
df[NOUVELLES_FEATURES].describe()


Valeurs infinies neutralisées : 0


,Ratio_Dette_Revenu,Ratio_EMI_Salaire,Taux_Investissement,Nb_produits
count,100000.000000,100000.000000,100000.000000,100000.000000
mean,0.060648,0.603104,0.052211,14.435630
std,0.087410,4.971237,0.031756,5.652887
min,0.000003,0.000000,0.000000,1.000000
25%,0.009571,0.012593,0.025935,10.000000
50%,0.028250,0.025257,0.044440,14.000000
75%,0.069712,0.042725,0.073231,19.000000
max,0.683252,227.739318,0.154430,32.000000


### Élasticité des revenus : `CV_Balance` par client

Le coefficient de variation de `Monthly_Balance` (écart-type / moyenne, calculé
sur les 8 mois de chaque client) mesure la stabilité du solde mensuel — c'est le
proxy d'élasticité des revenus identifié lors de l'EDA. Cette feature est
intrinsèquement **par client** ; elle est calculée ici puis reportée sur chaque
ligne mensuelle du client correspondant, avant d'être agrégée comme les autres
variables numériques à l'étape suivante.


In [4]:
stats_balance = df.groupby('Customer_ID')['Monthly_Balance'].agg(['mean', 'std'])
stats_balance['CV_Balance'] = stats_balance['std'] / stats_balance['mean']

# Repli sur la médiane globale pour les cas dégénérés (moyenne <= 0 ou valeur manquante)
mediane_cv = stats_balance.loc[stats_balance['mean'] > 0, 'CV_Balance'].median()
cas_degeneres = (stats_balance['mean'] <= 0) | stats_balance['CV_Balance'].isna()
print(f"Clients avec un cas dégénéré (moyenne <= 0 ou CV manquant) : {int(cas_degeneres.sum())}")
stats_balance.loc[cas_degeneres, 'CV_Balance'] = mediane_cv

df = df.merge(stats_balance['CV_Balance'], on='Customer_ID', how='left')
df['CV_Balance'].describe()


Clients avec un cas dégénéré (moyenne <= 0 ou CV manquant) : 0


count    100000.000000
mean          0.236102
std           0.138436
min           0.016191
25%           0.127512
50%           0.206493
75%           0.322515
max           1.057738
Name: CV_Balance, dtype: float64

### Élasticité des revenus v2 : régression log-linéaire

**Suite à la revue avec l'encadrant**, le `CV_Balance` ci-dessus s'avère
insuffisant comme mesure d'élasticité, pour deux raisons :

- **Trop d'erreur d'estimation** : le CV est calculé sur seulement 8 points par
  client (8 mois) — la moyenne et l'écart-type sont des estimateurs bruités sur
  un si petit échantillon.
- **Il confond croissance et instabilité** : un client en croissance
  *régulière* (par exemple +10 %/mois, son solde double sur la période) obtient
  un CV élevé simplement parce que sa moyenne évolue avec la tendance — le même
  CV qu'un client dont le solde varie de façon réellement chaotique d'un mois à
  l'autre. Le CV ne fait pas la différence entre les deux. Il explose en plus
  numériquement quand la moyenne du solde est petite (division par une valeur
  proche de 0), sans rapport avec la vraie variabilité du client.

On remplace l'élasticité par une **décomposition tendance / volatilité**, une
approche standard en finance pour ce type de série temporelle courte : pour
chaque client, on ajuste par régression linéaire (moindres carrés,
`np.polyfit`) la droite

`ln(solde_t) = alpha + beta * t`,  avec `t = 0, 1, ..., 7` (indice du mois)

Le passage au logarithme transforme les variations absolues en variations
**relatives** (en %), directement comparables entre un client modeste et un
client aisé — un solde qui passe de 1 000 à 1 100 (+10 %) et un solde qui passe
de 100 000 à 110 000 (+10 %) ont le même effet sur `ln(solde)`. On en tire deux
features complémentaires par client :

- **`Elasticite_Tendance`** (= `beta`) : le taux de croissance mensuel moyen du
  solde, en log donc approximativement en % — c'est la définition économique
  usuelle de l'élasticité (une tendance, pas une dispersion).
- **`Volatilite_Residuelle`** (= écart-type des résidus autour de la droite
  ajustée, `ddof=1`) : l'instabilité *réelle* du client une fois la tendance
  retirée — le vrai bruit, qui ne confond plus un client en croissance stable
  avec un client chaotique.

Un solde négatif ou nul rend le logarithme impossible : ces valeurs sont mises
à `NaN`. Un client avec moins de 3 points valides ne permet pas d'ajuster une
droite de façon fiable et reçoit `NaN` sur les deux features (imputé ensuite
par la médiane).

In [5]:
MONTH_ORDER = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August']
MONTH_NUM = {mois: i + 1 for i, mois in enumerate(MONTH_ORDER)}
df['Month_num'] = df['Month'].map(MONTH_NUM)
df = df.sort_values(['Customer_ID', 'Month_num'])

def regression_log_lineaire(groupe):
    """Ajuste ln(Monthly_Balance) = alpha + beta * t (t = 0..7) par MCO sur les
    mois où le solde est strictement positif. Renvoie (beta, écart-type des
    résidus) si au moins 3 points valides, sinon NaN sur les deux."""
    solde = groupe['Monthly_Balance'].to_numpy()
    t = np.arange(len(solde))
    valide = solde > 0
    if valide.sum() < 3:
        return pd.Series({'Elasticite_Tendance': np.nan, 'Volatilite_Residuelle': np.nan})

    log_solde = np.log(solde[valide])
    t_valide = t[valide]
    beta, alpha = np.polyfit(t_valide, log_solde, 1)
    residus = log_solde - (alpha + beta * t_valide)
    return pd.Series({
        'Elasticite_Tendance': beta,
        'Volatilite_Residuelle': residus.std(ddof=1),
    })

elasticite_v2 = (
    df.groupby('Customer_ID')[['Monthly_Balance']]
    .apply(regression_log_lineaire)
    .reset_index()
)

df = df.merge(elasticite_v2, on='Customer_ID', how='left')

print(f"Clients traités : {elasticite_v2.shape[0]}")
elasticite_v2[['Elasticite_Tendance', 'Volatilite_Residuelle']].describe()

Clients traités : 12500


,Elasticite_Tendance,Volatilite_Residuelle
count,12500.000000,12500.000000
mean,-0.000269,0.287927
std,0.063247,0.258322
min,-0.549253,0.010920
25%,-0.024238,0.120083
50%,-0.000056,0.208326
75%,0.023183,0.366670
max,0.687199,3.904802


In [6]:
n_clients_nan = int(elasticite_v2['Elasticite_Tendance'].isna().sum())
print(f"Clients avec moins de 3 points valides (NaN sur les 2 features) : {n_clients_nan}")

mediane_tendance = df['Elasticite_Tendance'].median()
mediane_volatilite = df['Volatilite_Residuelle'].median()
df['Elasticite_Tendance'] = df['Elasticite_Tendance'].fillna(mediane_tendance)
df['Volatilite_Residuelle'] = df['Volatilite_Residuelle'].fillna(mediane_volatilite)

print(f"Valeurs manquantes après imputation par la médiane : "
      f"{df[['Elasticite_Tendance', 'Volatilite_Residuelle']].isnull().sum().sum()}")
df[['Elasticite_Tendance', 'Volatilite_Residuelle']].describe()

Clients avec moins de 3 points valides (NaN sur les 2 features) : 0
Valeurs manquantes après imputation par la médiane : 0


,Elasticite_Tendance,Volatilite_Residuelle
count,100000.000000,100000.000000
mean,-0.000269,0.287927
std,0.063245,0.258313
min,-0.549253,0.010920
25%,-0.024238,0.120083
50%,-0.000056,0.208326
75%,0.023183,0.366670
max,0.687199,3.904802


In [7]:
profil_elasticite = df[['Customer_ID', 'CV_Balance', 'Elasticite_Tendance', 'Volatilite_Residuelle']].drop_duplicates('Customer_ID')
correlation_elasticite = profil_elasticite[['CV_Balance', 'Elasticite_Tendance', 'Volatilite_Residuelle']].corr()
correlation_elasticite.round(3)

,CV_Balance,Elasticite_Tendance,Volatilite_Residuelle
CV_Balance,1.000,-0.003,0.878
Elasticite_Tendance,-0.003,1.000,0.008
Volatilite_Residuelle,0.878,0.008,1.000


**Interprétation** : `CV_Balance` corrèle fortement avec `Volatilite_Residuelle`
(0.878) mais quasiment pas avec `Elasticite_Tendance` (-0.003) — l'ancien CV
mesurait donc essentiellement le bruit résiduel, sans capter la tendance des
clients. `Elasticite_Tendance` et `Volatilite_Residuelle` sont quant à elles
quasi indépendantes l'une de l'autre (0.008) : la régression log-linéaire
sépare effectivement deux dimensions distinctes du comportement client
(croissance vs instabilité), là où le CV les mélangeait en un seul chiffre.

### Binarisation multi-label de `Type_of_Loan`

`Type_of_Loan` liste, pour chaque relevé mensuel, les types de prêts détenus par
le client sous forme de texte libre (ex. `"Auto Loan, Credit-Builder Loan,
Personal Loan, and Home Equity Loan"`), avec parfois des doublons et un `and`
avant le dernier élément. Ce format n'est pas exploitable tel quel par un modèle.

On le transforme en **9 colonnes binaires** (une par type de prêt possible) :
`Auto Loan`, `Personal Loan`, `Credit-Builder Loan`, `Home Equity Loan`,
`Mortgage Loan`, `Student Loan`, `Debt Consolidation Loan`, `Payday Loan`, et
`No Loan`. Un client vaut `1` sur une colonne s'il a détenu ce type de prêt sur
**au moins un des 8 mois observés** (union des types rencontrés sur toute la
période, pas seulement le mois le plus fréquent) ; `No Loan` vaut `1` uniquement
si aucun des 8 autres types n'a jamais été détecté pour ce client.


In [8]:
TYPES_PRET_CIBLES = [
    'Auto Loan', 'Personal Loan', 'Credit-Builder Loan', 'Home Equity Loan',
    'Mortgage Loan', 'Student Loan', 'Debt Consolidation Loan', 'Payday Loan',
]

def parser_types_prets(valeur):
    """Découpe le texte libre de Type_of_Loan en un ensemble de types de prêts."""
    if pd.isna(valeur):
        return set()
    items = [x.strip() for x in str(valeur).split(',')]
    items = [x[4:].strip() if x.lower().startswith('and ') else x for x in items]
    return set(items)

types_par_ligne = df['Type_of_Loan'].apply(parser_types_prets)
types_par_client = types_par_ligne.groupby(df['Customer_ID']).agg(lambda s: set.union(*s))

COLONNES_PRETS = ['Loan_' + t.replace(' ', '_').replace('-', '_') for t in TYPES_PRET_CIBLES]

loan_features = pd.DataFrame(index=types_par_client.index)
for type_pret, colonne in zip(TYPES_PRET_CIBLES, COLONNES_PRETS):
    loan_features[colonne] = types_par_client.apply(lambda s, t=type_pret: int(t in s))

loan_features['Loan_No_Loan'] = (loan_features[COLONNES_PRETS].sum(axis=1) == 0).astype(int)

print("Nombre de clients détenant chaque type de prêt :")
loan_features.sum().sort_values(ascending=False)


Nombre de clients détenant chaque type de prêt :


Loan_Payday_Loan                3993
Loan_Credit_Builder_Loan        3966
Loan_Home_Equity_Loan           3925
Loan_Mortgage_Loan              3920
Loan_Personal_Loan              3888
Loan_Debt_Consolidation_Loan    3880
Loan_Student_Loan               3880
Loan_Auto_Loan                  3820
Loan_No_Loan                    1621
dtype: int64

## 3. Agrégation par `Customer_ID`

On passe de 8 lignes mensuelles par client à **une seule ligne par client** :

- **Moyenne** pour les variables numériques (comportement moyen sur la période
  observée).
- **Mode** (valeur la plus fréquente) pour les variables catégorielles
  (catégorie la plus représentative du client).
- **Dernière valeur chronologique** (mois d'août, le plus récent) pour la cible
  `Credit_Score`, qui reflète le statut de crédit le plus à jour du client.


In [9]:
MONTH_ORDER = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August']
df['Month'] = pd.Categorical(df['Month'], categories=MONTH_ORDER, ordered=True)
df = df.sort_values(['Customer_ID', 'Month'])

COLONNES_NUMERIQUES = [
    'Age', 'Annual_Income', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts',
    'Num_Credit_Card', 'Interest_Rate', 'Num_of_Loan', 'Delay_from_due_date',
    'Num_of_Delayed_Payment', 'Changed_Credit_Limit', 'Num_Credit_Inquiries',
    'Outstanding_Debt', 'Credit_Utilization_Ratio', 'Total_EMI_per_month',
    'Amount_invested_monthly', 'Monthly_Balance', 'Credit_History_Age_Months',
    'Ratio_Dette_Revenu', 'Ratio_EMI_Salaire', 'Taux_Investissement',
    'Nb_produits', 'CV_Balance', 'Elasticite_Tendance', 'Volatilite_Residuelle',
]
COLONNES_CATEGORIELLES = ['Occupation', 'Credit_Mix', 'Payment_of_Min_Amount', 'Payment_Behaviour']

def mode_ou_nan(s):
    m = s.mode(dropna=True)
    return m.iloc[0] if not m.empty else np.nan

agg_numerique = df.groupby('Customer_ID')[COLONNES_NUMERIQUES].mean()
agg_categorielle = df.groupby('Customer_ID')[COLONNES_CATEGORIELLES].agg(mode_ou_nan)
agg_cible = df.groupby('Customer_ID')['Credit_Score'].last()

df_client = (
    agg_numerique
    .join(agg_categorielle)
    .join(agg_cible.rename('Credit_Score'))
    .join(loan_features)
    .reset_index()
)

print(f"Dimensions après agrégation : {df_client.shape[0]} lignes x {df_client.shape[1]} colonnes")
df_client.head()

Dimensions après agrégation : 12500 lignes x 39 colonnes


,Customer_ID,Age,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,Num_Credit_Inquiries,Outstanding_Debt,Credit_Utilization_Ratio,Total_EMI_per_month,Amount_invested_monthly,Monthly_Balance,Credit_History_Age_Months,Ratio_Dette_Revenu,Ratio_EMI_Salaire,Taux_Investissement,Nb_produits,CV_Balance,Elasticite_Tendance,Volatilite_Residuelle,Occupation,Credit_Mix,Payment_of_Min_Amount,Payment_Behaviour,Credit_Score,Loan_Auto_Loan,Loan_Personal_Loan,Loan_Credit_Builder_Loan,Loan_Home_Equity_Loan,Loan_Mortgage_Loan,Loan_Student_Loan,Loan_Debt_Consolidation_Loan,Loan_Payday_Loan,Loan_No_Loan
0,CUS_0x1000,18.000,30625.94,2706.161667,6.0,5.0,27.0,2.0,62.25,25.000,1.880,10.875,1562.91,33.477546,42.941090,158.549735,335.375341,125.500,0.051032,0.015868,0.058588,13.0,0.192581,0.022070,0.191426,Lawyer,Bad,Yes,High_spent_Medium_value_payments,Poor,0,0,1,1,0,0,0,0,0
1,CUS_0x1009,25.750,52312.68,4250.390000,6.0,5.0,17.0,4.0,7.25,234.250,9.730,3.000,202.68,29.839984,108.366467,146.679378,428.743155,368.375,0.003874,0.025496,0.034510,15.0,0.128525,0.038710,0.090053,Mechanic,Standard,Yes,High_spent_Medium_value_payments,Standard,0,0,1,1,0,0,0,1,0
2,CUS_0x100b,18.500,113781.39,9549.782500,1.0,4.0,1.0,0.0,13.50,7.375,10.965,1.750,1030.20,34.841449,0.000000,509.175204,781.229776,186.375,0.009054,0.000000,0.053318,5.0,0.263905,-0.007052,0.282585,Media_Manager,Good,No,High_spent_Large_value_payments,Standard,0,0,0,0,0,0,0,0,1
3,CUS_0x1011,43.875,58918.47,5208.872500,3.0,3.0,17.0,3.0,27.25,14.625,14.170,7.000,473.14,27.655897,123.434939,320.097156,332.642837,186.500,0.008030,0.023697,0.061452,9.0,0.576030,-0.159830,0.463012,Doctor,Standard,Yes,Low_spent_Medium_value_payments,Standard,0,0,1,0,0,1,1,0,0
4,CUS_0x1013,43.750,98620.98,7962.415000,3.0,3.0,6.0,3.0,12.50,8.500,1.705,3.000,1233.51,31.933940,6266.765823,355.442408,472.781009,210.375,0.012508,0.787043,0.044640,9.0,0.383886,-0.046076,0.588964,Mechanic,Good,No,High_spent_Medium_value_payments,Standard,0,1,0,0,0,1,1,0,0


## 4. Encodage des variables catégorielles

- **`Credit_Mix`** : label encoding ordinal, car les catégories ont un ordre
  naturel de qualité de crédit : `Bad` (0) < `Standard` (1) < `Good` (2).
- **`Payment_of_Min_Amount`** : binaire (`No` = 0, `Yes` = 1) — deux modalités
  seulement, un label encoding suffit (équivalent à un one-hot à une colonne).
- **`Occupation`** et **`Payment_Behaviour`** : one-hot encoding, car ce sont des
  catégories nominales sans ordre naturel.

`Credit_Score` (la cible) reste inchangée sous forme de texte (`Good` /
`Standard` / `Poor`) : son encodage relève du notebook de modélisation, pas du
feature engineering.


In [10]:
CREDIT_MIX_ORDRE = {'Bad': 0, 'Standard': 1, 'Good': 2}
df_client['Credit_Mix'] = df_client['Credit_Mix'].map(CREDIT_MIX_ORDRE)

PAYMENT_MIN_ORDRE = {'No': 0, 'Yes': 1}
df_client['Payment_of_Min_Amount'] = df_client['Payment_of_Min_Amount'].map(PAYMENT_MIN_ORDRE)

print("Valeurs manquantes après encodage ordinal/binaire :")
print(df_client[['Credit_Mix', 'Payment_of_Min_Amount']].isnull().sum())


Valeurs manquantes après encodage ordinal/binaire :
Credit_Mix               0
Payment_of_Min_Amount    0
dtype: int64


In [11]:
nb_colonnes_avant = df_client.shape[1]

df_client = pd.get_dummies(
    df_client, columns=['Occupation', 'Payment_Behaviour'],
    prefix=['Occupation', 'Payment_Behaviour'],
)

colonnes_dummies = [c for c in df_client.columns if c.startswith('Occupation_') or c.startswith('Payment_Behaviour_')]
df_client[colonnes_dummies] = df_client[colonnes_dummies].astype(int)

print(f"Colonnes avant one-hot : {nb_colonnes_avant}")
print(f"Colonnes après one-hot : {df_client.shape[1]} (+{len(colonnes_dummies)} colonnes indicatrices)")


Colonnes avant one-hot : 39
Colonnes après one-hot : 58 (+21 colonnes indicatrices)


## 5. Suppression des colonnes inutiles

`ID`, `Name`, `SSN` et `Month` sont déjà absentes de `df_client` : elles n'ont
jamais été incluses dans l'agrégation par client (identifiants ou texte libre
sans valeur prédictive). Il reste à supprimer :

- **`Customer_ID`** : identifiant, non prédictif une fois l'agrégation terminée.
- **`Monthly_Inhand_Salary`** : quasi colinéaire avec `Annual_Income` (identifié
  lors de l'EDA, confirmé ci-dessous), et déjà exploitée pour construire
  `Ratio_EMI_Salaire` et `Taux_Investissement`.


In [12]:
correlation_revenu = df_client[['Annual_Income', 'Monthly_Inhand_Salary']].corr().iloc[0, 1]
print(f"Corrélation Annual_Income / Monthly_Inhand_Salary : {correlation_revenu:.3f}")

COLONNES_A_SUPPRIMER = [c for c in ['ID', 'Customer_ID', 'Name', 'SSN', 'Month', 'Monthly_Inhand_Salary']
                         if c in df_client.columns]
print(f"Colonnes supprimées : {COLONNES_A_SUPPRIMER}")

df_client = df_client.drop(columns=COLONNES_A_SUPPRIMER)
print(f"Dimensions après suppression : {df_client.shape[0]} lignes x {df_client.shape[1]} colonnes")


Corrélation Annual_Income / Monthly_Inhand_Salary : 0.998
Colonnes supprimées : ['Customer_ID', 'Monthly_Inhand_Salary']
Dimensions après suppression : 12500 lignes x 56 colonnes


## 6. Vérification finale et export

On vérifie l'absence de valeurs manquantes, les types de données, et un aperçu
statistique avant d'exporter la table finale.


In [13]:
missing = df_client.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
if missing.empty:
    print("Aucune valeur manquante.")
else:
    print("Valeurs manquantes restantes :")
    print(missing)


Aucune valeur manquante.


In [14]:
df_client.dtypes


Age                                                   float64
Annual_Income                                         float64
Num_Bank_Accounts                                     float64
Num_Credit_Card                                       float64
Interest_Rate                                         float64
Num_of_Loan                                           float64
Delay_from_due_date                                   float64
Num_of_Delayed_Payment                                float64
Changed_Credit_Limit                                  float64
Num_Credit_Inquiries                                  float64
Outstanding_Debt                                      float64
Credit_Utilization_Ratio                              float64
Total_EMI_per_month                                   float64
Amount_invested_monthly                               float64
Monthly_Balance                                       float64
Credit_History_Age_Months                             float64
Ratio_De

In [15]:
print(f"Dimensions finales : {df_client.shape[0]} lignes x {df_client.shape[1]} colonnes")
print(f"Nombre de clients uniques attendu : 12500 -> obtenu : {df_client.shape[0]}")
df_client.describe().T


Dimensions finales : 12500 lignes x 56 colonnes
Nombre de clients uniques attendu : 12500 -> obtenu : 12500


,count,mean,std,min,25%,50%,75%,max
Age,12500.0,34.337570,9.854151,18.000000,26.125000,34.000000,41.750000,56.500000
Annual_Income,12500.0,50518.846732,38300.174697,7005.930000,19358.250000,37037.320000,71689.865000,179987.280000
Num_Bank_Accounts,12500.0,5.368840,2.592483,0.000000,3.000000,5.375000,7.000000,10.500000
Num_Credit_Card,12500.0,5.533910,2.066255,0.500000,4.000000,5.000000,7.000000,10.875000
Interest_Rate,12500.0,14.532080,8.741636,1.000000,7.000000,13.000000,20.000000,34.000000
Num_of_Loan,12500.0,3.532880,2.446442,0.000000,2.000000,3.000000,5.000000,9.000000
Delay_from_due_date,12500.0,21.068780,14.772965,-2.000000,9.875000,17.875000,28.000000,63.250000
Num_of_Delayed_Payment,12500.0,29.693800,77.254630,-1.250000,9.250000,14.500000,19.000000,973.250000
Changed_Credit_Limit,12500.0,10.389113,6.541147,-1.070000,5.450000,9.370000,14.660000,31.115000
Num_Credit_Inquiries,12500.0,5.779590,3.710482,0.000000,3.000000,5.250000,8.500000,16.375000


In [16]:
df_client['Credit_Score'].value_counts()


Credit_Score
Standard    6485
Poor        3602
Good        2413
Name: count, dtype: int64

## 7. Export

Export de la table finale, une ligne par client, prête pour la modélisation.


In [17]:
df_client.to_csv('../data/train_features.csv', index=False)
print("Fichier exporté avec succès : data/train_features.csv")
print(f"Dimensions exportées : {df_client.shape[0]} lignes x {df_client.shape[1]} colonnes")


Fichier exporté avec succès : data/train_features.csv
Dimensions exportées : 12500 lignes x 56 colonnes
